# DataDrivenNN SoH — final result analysis

This notebook presents the frozen Step 9 baseline test results produced by three fixed seeds. It does **not** load the battery dataset, construct a model, or train anything. Metrics and figures come from the saved read-only analysis artifacts.

In [ ]:
import csv
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ANALYSIS = ROOT / 'artifacts' / 'data_driven' / 'analysis'
FIGURES = ANALYSIS / 'figures'
summary = json.loads((ANALYSIS / 'analysis_summary.json').read_text())
summary['seeds'], summary['records'], summary['batteries']

## Headline test metrics

The ensemble is the equal-weight mean of predictions aligned using `(battery_id, cycle)`. Mean ± SD describes variation across independently initialized models; the ensemble row describes their averaged prediction.

In [ ]:
m = summary['ensemble_metrics']
across = summary['across_seed_metrics']
table = f'''
| Result | MAE | RMSE | R² | Bias |
|---|---:|---:|---:|---:|
| Three-seed ensemble | {m['mae']:.5f} | {m['rmse']:.5f} | {m['r2']:.3f} | {m['bias']:.5f} |
| Seed mean ± sample SD | {across['mae']['mean']:.5f} ± {across['mae']['sample_std']:.5f} | {across['rmse']['mean']:.5f} ± {across['rmse']['sample_std']:.5f} | {across['r2']['mean']:.3f} ± {across['r2']['sample_std']:.3f} | {across['bias']['mean']:.5f} ± {across['bias']['sample_std']:.5f} |
'''
display(Markdown(table))

A positive bias means the baseline tends to predict SoH above the measured value. This should be stated alongside RMSE and R² because optimistic SoH estimates matter in battery prognostics.

In [ ]:
display(Image(filename=str(FIGURES / '01_actual_vs_predicted_soh.png'), width=800))

## Battery degradation trajectories

The batteries below are selected reproducibly as the lowest-, median-, and highest-RMSE batteries. The shaded band is ±1 standard deviation across seeds; it is not a fitted confidence interval.

In [ ]:
display(Image(filename=str(FIGURES / '02_representative_battery_trajectories.png'), width=1100))

## Error and seed stability

In [ ]:
for name in ('03_per_battery_errors', '04_error_distributions', '06_seed_stability'):
    display(Image(filename=str(FIGURES / f'{name}.png'), width=1000))

## Optimization history

Validation data loss is solid and training data loss is dashed. Dotted lines mark the validation-selected checkpoint for each seed. The baseline has no physics loss or learned dynamics, so no physics diagnostics are shown.

In [ ]:
display(Image(filename=str(FIGURES / '05_training_curves.png'), width=1000))

## Per-battery table

In [ ]:
with (ANALYSIS / 'per_battery_metrics.csv').open() as stream:
    batteries = list(csv.DictReader(stream))
worst = sorted(batteries, key=lambda row: float(row['rmse']), reverse=True)[:10]
rows = ['| Battery | Records | MAE | RMSE | R² | Bias |', '|---:|---:|---:|---:|---:|---:|']
for row in worst:
    rows.append(f"| {row['battery_id']} | {row['records']} | {float(row['mae']):.5f} | {float(row['rmse']):.5f} | {float(row['r2']):.3f} | {float(row['bias']):.5f} |")
display(Markdown('### Ten highest-RMSE batteries\n\n' + '\n'.join(rows)))

## Presentation guidance

For a standalone baseline-results slide, use measured versus predicted SoH, representative trajectories, and the ensemble metrics. Use per-battery errors or seed stability as supporting evidence. SVG and PDF versions are available for lossless slide placement. Cross-model claims should wait for the shared comparison analysis so every method is evaluated identically.